In [ ]:
# ============================================================
# Phase 3: Artificial Neural Network (ANN) Model
# Objective: To build a non-linear deep learning model for predicting
# early-stage Alzheimer's Disease using tabular data.
# ============================================================

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.metrics import roc_curve, auc, precision_recall_curve

import plotly.express as px

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

In [4]:
df = pd.read_csv('/content/Alzheimer_final.csv')
df.head()

,ID,Age,Gender,Educ,MMSE,Target,eTIV,nWBV
0,011_S_0003,0.717371,1,4.0,0.081752,1.0,0.014743,0.223724
1,022_S_0004,-0.854411,1,0.0,0.836511,1.0,0.014743,0.223724
2,011_S_0005,-0.148249,1,3.0,1.052156,0.0,0.014743,0.223724
3,100_S_0006,0.614863,0,2.0,0.620865,1.0,0.014743,0.223724
4,011_S_0010,-0.125469,0,1.0,0.513043,1.0,0.014743,0.223724


In [5]:
X = df.drop(columns=['ID', 'Target'])
y = df['Target'].astype(int)

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

In [7]:
ann_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    BatchNormalization(),
    Dropout(0.3),

    Dense(32, activation='relu'),
    BatchNormalization(),
    Dropout(0.2),

    Dense(16, activation='relu'),
    Dropout(0.1),

    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
ann_model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall'),
        tf.keras.metrics.AUC(name='auc')
    ]
)

ann_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,457 (13.50 KB)

 Trainable params: 3,265 (12.75 KB)

 Non-trainable params: 192 (768.00 B)

In [12]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True
)

history = ann_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=150,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/150
60/60 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.5468 - auc: 0.5279 - loss: 0.7384 - precision: 0.4668 - recall: 0.4166 - val_accuracy: 0.6025 - val_auc: 0.5807 - val_loss: 0.6767 - val_precision: 0.6075 - val_recall: 0.3081
Epoch 2/150
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5870 - auc: 0.5958 - loss: 0.6807 - precision: 0.5244 - recall: 0.3721 - val_accuracy: 0.5920 - val_auc: 0.6220 - val_loss: 0.6680 - val_precision: 0.6406 - val_recall: 0.1943
Epoch 3/150
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5849 - auc: 0.6106 - loss: 0.6699 - precision: 0.5208 - recall: 0.3721 - val_accuracy: 0.6131 - val_auc: 0.6655 - val_loss: 0.6561 - val_precision: 0.7059 - val_recall: 0.2275
Epoch 4/150
60/60 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5880 - auc: 0.5999 - loss: 0.6749 - precision: 0.5239 - recall: 0.4067 - val_accuracy: 0.6279 - val_auc: 0.6800 - val_loss: 0.6461 - val_precision: 0.7108 - val_recall: 0.2796
Epoch 5/150
60/60 ━━━━━━━━━━━━━━━━━

In [14]:
test_loss, test_accuracy, test_precision, test_recall, test_auc = ann_model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)
print("Test Precision:", test_precision)
print("Test Recall:", test_recall)
print("Test AUC:", test_auc)

Test Loss: 0.5855701565742493
Test Accuracy: 0.6646943092346191
Test Precision: 0.7355769276618958
Test Recall: 0.3493150770664215
Test AUC: 0.7093876600265503


In [15]:
y_prob = ann_model.predict(X_test).ravel()
y_pred = (y_prob >= 0.50).astype(int)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [16]:
print("--- ANN Baseline Results ---\n")
print(classification_report(
    y_test,
    y_pred,
    target_names=['Healthy (0)', 'Early-Stage (1)']
))

--- ANN Baseline Results ---

                 precision    recall  f1-score   support

    Healthy (0)       0.65      0.90      0.75       576
Early-Stage (1)       0.74      0.35      0.47       438

       accuracy                           0.66      1014
      macro avg       0.69      0.63      0.61      1014
   weighted avg       0.68      0.66      0.63      1014



In [17]:
cm = confusion_matrix(y_test, y_pred)

fig_cm = px.imshow(
    cm,
    text_auto=True,
    color_continuous_scale='Blues',
    labels=dict(
        x="Predicted Diagnosis",
        y="Actual Diagnosis",
        color="Patients"
    ),
    x=['Healthy (0)', 'Early-Stage (1)'],
    y=['Healthy (0)', 'Early-Stage (1)'],
    title="Interactive Confusion Matrix"
)

fig_cm.update_layout(
    title_x=0.5,
    width=600,
    height=600
)

fig_cm.show()

In [18]:
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

fig_roc = px.area(
    x=fpr,
    y=tpr,
    title=f'Interactive ROC Curve (AUC = {roc_auc:.4f})',
    labels=dict(
        x='False Positive Rate (1 - Specificity)',
        y='True Positive Rate (Sensitivity/Recall)'
    ),
    width=700,
    height=600,
    color_discrete_sequence=['#1f77b4']
)

fig_roc.add_shape(
    type='line',
    line=dict(dash='dash', color='gray'),
    x0=0,
    x1=1,
    y0=0,
    y1=1
)

fig_roc.update_layout(title_x=0.5)
fig_roc.show()

In [19]:
custom_threshold = 0.40

y_pred_medical = (y_prob >= custom_threshold).astype(int)

print(f"--- Results with {custom_threshold*100}% Threshold ---")
print(classification_report(
    y_test,
    y_pred_medical,
    target_names=['Healthy (0)', 'Early-Stage (1)']
))

--- Results with 40.0% Threshold ---
                 precision    recall  f1-score   support

    Healthy (0)       0.74      0.46      0.57       576
Early-Stage (1)       0.53      0.79      0.63       438

       accuracy                           0.60      1014
      macro avg       0.63      0.62      0.60      1014
   weighted avg       0.65      0.60      0.59      1014



In [20]:
cm_40 = confusion_matrix(y_test, y_pred_medical)

fig_cm = px.imshow(
    cm_40,
    text_auto=True,
    color_continuous_scale='blues',
    labels=dict(
        x="Predicted Diagnosis",
        y="Actual Diagnosis",
        color="Patients"
    ),
    x=['Healthy (0)', 'Early-Stage (1)'],
    y=['Healthy (0)', 'Early-Stage (1)'],
    title="Confusion Matrix at 40% Threshold"
)

fig_cm.update_layout(
    title_x=0.5,
    width=650,
    height=600
)

fig_cm.show()

In [21]:
precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob)

tradeoff_df = pd.DataFrame({
    'Threshold': thresholds,
    'Precision': precisions[:-1],
    'Recall': recalls[:-1]
})

tradeoff_melted = tradeoff_df.melt(
    id_vars='Threshold',
    value_vars=['Precision', 'Recall'],
    var_name='Metric',
    value_name='Score'
)

In [22]:
fig_tradeoff = px.line(
    tradeoff_melted,
    x='Threshold',
    y='Score',
    color='Metric',
    title="Precision-Recall Trade-Off Analysis",
    labels={'Score': 'Percentage (0 to 1)'},
    color_discrete_sequence=['#ff7f0e', '#1f77b4']
)

fig_tradeoff.add_vline(
    x=0.40,
    line_width=3,
    line_dash="dash",
    line_color="red",
    annotation_text="Our 40% Medical Target",
    annotation_position="top right"
)

fig_tradeoff.update_layout(
    title_x=0.5,
    width=800,
    height=500
)

fig_tradeoff.show()

In [23]:
history_df = pd.DataFrame(history.history)
history_df['Epoch'] = range(1, len(history_df) + 1)

In [24]:
fig_acc = px.line(
    history_df,
    x='Epoch',
    y=['accuracy', 'val_accuracy'],
    title='ANN Training vs Validation Accuracy',
    labels={
        'value': 'Accuracy',
        'variable': 'Metric'
    }
)

fig_acc.update_layout(
    title_x=0.5,
    width=800,
    height=500
)

fig_acc.show()

In [25]:
fig_loss = px.line(
    history_df,
    x='Epoch',
    y=['loss', 'val_loss'],
    title='ANN Training vs Validation Loss',
    labels={
        'value': 'Loss',
        'variable': 'Metric'
    }
)

fig_loss.update_layout(
    title_x=0.5,
    width=800,
    height=500
)

fig_loss.show()

In [26]:
fig_auc = px.line(
    history_df,
    x='Epoch',
    y=['auc', 'val_auc'],
    title='ANN Training vs Validation AUC',
    labels={
        'value': 'AUC',
        'variable': 'Metric'
    }
)

fig_auc.update_layout(
    title_x=0.5,
    width=800,
    height=500
)

fig_auc.show()